# PneumoScan AI — Training (Colab / Kaggle GPU)

Trains **DenseNet121** and **Swin-Tiny** for Stage 1 (Normal vs Pneumonia) and Stage 2 (Bacterial vs Viral)
with identical settings, then calibrates them. Outputs go to `backend/weights/` and `backend/metrics/metrics.json`.

**Runtime → Change runtime type → GPU** before starting.

> Data use: the Kermany dataset is public (CC BY 4.0). Any testing with real patient images requires consent and
> appropriate institutional approval.

## 1. Get the code
Upload/clone the project repository (replace the URL with yours), or upload a zip and unzip it.

In [ ]:
import os
REPO_URL = "https://github.com/<your-org>/pneumoscan-ai.git"  # <- change me
if not os.path.exists("pneumoscan-ai"):
    !git clone -q $REPO_URL pneumoscan-ai
%cd pneumoscan-ai/training
!pip -q install -r requirements.txt

## 2. Download the Kermany dataset
**Colab:** upload your `kaggle.json` API token (Kaggle → Account → Create New Token).
**Kaggle notebooks:** add the dataset *paultimothymooney/chest-xray-pneumonia* as input instead and set `DATA_ROOT`
to `/kaggle/input/chest-xray-pneumonia/chest_xray`.

In [ ]:
DATA_ROOT = "/content/data/chest_xray"
if not os.path.exists(DATA_ROOT):
    from google.colab import files  # noqa
    files.upload()  # select kaggle.json
    !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
    !kaggle datasets download -q -d paultimothymooney/chest-xray-pneumonia -p /content/data
    !unzip -q -o /content/data/chest-xray-pneumonia.zip -d /content/data

## 3. Patient-grouped 70/15/15 split
Re-splits all images (the 16-image Kaggle `val` folder is not used) and removes duplicate files.

In [ ]:
!python prepare_data.py --data-root $DATA_ROOT --out data/splits.csv

## 4. Train both architectures, both stages (identical hyper-parameters)

In [ ]:
EPOCHS = 25
for arch in ["densenet", "swin"]:
    for task in ["stage1", "stage2"]:
        !python train.py --arch $arch --task $task --epochs $EPOCHS --batch-size 32 --workers 2

Optional — single 3-class model for comparing approaches (`CLASSIFICATION_MODE=three_class` in the API):

In [ ]:
# for arch in ["densenet", "swin"]:
#     !python train.py --arch $arch --task three_class --epochs $EPOCHS

## 5. Temperature scaling on the validation set

In [ ]:
!python calibrate.py --all

## 6. Download the weights
Copy these files into `backend/weights/` on your machine and set `USE_MOCK_MODELS=false`.

In [ ]:
!cd .. && zip -qr /content/pneumoscan_weights.zip backend/weights backend/metrics
try:
    from google.colab import files
    files.download("/content/pneumoscan_weights.zip")
except ImportError:
    print("Weights zipped at /content/pneumoscan_weights.zip")